# SpatioCore Omics Function Testing Notebook

This notebook demonstrates the major `spatialcore` APIs with small synthetic data only. It is designed as a user-facing workflow example, not as a real analysis notebook.

Important behavior shown here:

- Plots display inline and return figure/axis objects.
- `save_path=None` is used throughout, so nothing is saved locally.
- Real Xenium, Visium HD, CODEX, H&E, AnnData, SpatialData, Zarr, H5AD, CSV, Parquet, Excel, and image files are not loaded or written.
- Optional workflow backends such as Squidpy, Voyager, SpatialData, CellCharter, PyDESeq2, gseapy, decoupler, and scCODA are demonstrated with guarded cells.

## Design Notes

The package is intentionally function-based at this stage. AnnData and SpatialData already hold most workflow state, and small functions are easier to test with synthetic data. Project classes such as `SpatioCoreProject`, `XeniumProject`, or `VisiumHDProject` may become useful later for path manifests, batch execution, and report orchestration, but they are deferred until the workflow boundaries stabilize.

Squidpy is now the preferred optional backend for spatial graph construction, co-occurrence curves, Ripley-style workflows, and global Moran's I. Local Moran's I is routed through Voyager when `voyagerpy.spatial.local_moran` is available.

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt

import anndata as ad

from spatialcore import (
    alignment,
    annotation,
    composition,
    de,
    enrichment,
    image,
    io,
    lr,
    niches,
    plotting,
    preprocessing,
    qc,
    reports,
    spatial,
)

plotting.apply_publication_style(base_font_size=8, dpi=300)

## Synthetic AnnData

This mock object contains spatial coordinates, sample metadata, cell types, conditions, a counts layer, and a small PCA-like representation. It is used to mimic Xenium and Visium HD workflows without loading any real files.

In [ ]:
rng = np.random.default_rng(7)
n_cells = 80
genes = ["mt-Nd1", "GeneA", "GeneB", "GeneC", "GeneD", "GeneE", "Ifng", "Col1a1"]

coords = np.column_stack([
    rng.normal(loc=np.repeat([0, 60, 120, 180], n_cells // 4), scale=8),
    rng.normal(loc=np.tile(np.repeat([0, 60], n_cells // 8), 4), scale=8),
])

cell_types = np.resize(["Tumor", "T cell", "Macrophage", "Hepatocyte"], n_cells)
conditions = np.where(np.arange(n_cells) % 2 == 0, "ctrl", "tx")
samples = np.where(np.arange(n_cells) < n_cells / 2, "sample_1", "sample_2")

counts = rng.poisson(lam=2.0, size=(n_cells, len(genes))).astype(float)
counts[cell_types == "T cell", genes.index("Ifng")] += rng.poisson(5, size=(cell_types == "T cell").sum())
counts[cell_types == "Hepatocyte", genes.index("Col1a1")] += rng.poisson(4, size=(cell_types == "Hepatocyte").sum())

obs = pd.DataFrame(
    {
        "cell_id": [f"physical_cell_{i // 2}" if i < 6 else f"physical_cell_{i}" for i in range(n_cells)],
        "sample": samples,
        "cell_type": cell_types,
        "condition": conditions,
        "timepoint": np.where(np.arange(n_cells) % 3 == 0, 4, 8),
        "batch_pair": np.where(samples == "sample_1", "batch_a", "batch_b"),
        "lesion": coords[:, 0] > 100,
    },
    index=[f"obs_{i}" for i in range(n_cells)],
)

adata = ad.AnnData(X=sparse.csr_matrix(counts), obs=obs, var=pd.DataFrame(index=genes))
adata.layers["counts"] = adata.X.copy()
adata.obsm["spatial"] = coords
adata.obsm["X_pca"] = rng.normal(size=(n_cells, 6))
adata.obsm["X_mock_niche"] = rng.normal(size=(n_cells, 4))

adata

## Xenium-Style Cleanup And QC

The reference notebooks often remove duplicate physical-cell rows, compute QC metrics, and inspect distributions and spatial QC patterns. The plotting functions display inline, return objects, and do not save because `save_path=None`.

In [ ]:
adata_one = io.one_row_per_cell(adata, cell_id_col="cell_id")
adata_qc = qc.flag_mito_genes(adata_one, prefix="mt-", inplace=False)
qc_df, qc_summary, qc_metadata = qc.compute_qc_metrics(adata_qc)

display(qc_metadata)
display(qc_summary)
display(qc_df.head())

fig_hist, axes_hist = qc.plot_qc_histograms(adata_qc, save_path=None, show=True)
fig_box, axes_box = qc.plot_qc_boxplots(adata_qc, sample_col="sample", save_path=None, show=True)
fig_qc_spatial, ax_qc_spatial = qc.plot_qc_spatial(adata_qc, color="total_counts", save_path=None, show=True)

## Publication-Style Spatial Plots

These examples mirror the notebook plotting style: small points, clean axes, readable legends, high-DPI defaults, and returned figure/axis handles.

In [ ]:
spatial_df = adata_qc.obs[["cell_type", "condition", "total_counts"]].copy()
spatial_df["x"] = adata_qc.obsm["spatial"][:, 0]
spatial_df["y"] = adata_qc.obsm["spatial"][:, 1]

fig_ct, ax_ct = plotting.plot_spatial_scatter(
    spatial_df,
    color="cell_type",
    categorical=True,
    title="Synthetic cell types",
    size=8,
    save_path=None,
    show=True,
)

fig_multi, axes_multi = plt.subplots(1, 2, figsize=(7, 3), dpi=300)
plotting.plot_spatial_scatter(spatial_df, color="cell_type", categorical=True, ax=axes_multi[0], fig=fig_multi, show=False)
plotting.plot_spatial_scatter(spatial_df, color="total_counts", categorical=False, ax=axes_multi[1], fig=fig_multi, show=False)
axes_multi[0].set_title("Cell type")
axes_multi[1].set_title("Total counts")
plotting.finalize_figure(fig_multi, save_path=None, show=True)

## Visium HD-Style Preprocessing

This section shows Voyager-like thresholding and Scanpy-style preprocessing on mock counts. It avoids reading Visium HD files and catches optional backend issues cleanly.

In [ ]:
adata_voyager = preprocessing.voyager_transform(adata_qc)
display(adata_voyager.uns["voyager_thresholds"])
display(adata_voyager.obs["voyager_keep"].value_counts())

try:
    adata_norm = preprocessing.normalize_log1p(adata_qc, inplace=False)
    adata_norm = preprocessing.select_hvgs(adata_norm, n_top_genes=5, flavor="seurat", subset=False)
    adata_norm = preprocessing.run_pca_neighbors(adata_norm, n_comps=4, n_neighbors=8, inplace=False)
    display(adata_norm.var[["highly_variable"]])
except Exception as exc:
    print(f"Skipped Scanpy preprocessing step in this environment: {exc}")

## KNN Neighbor And Leiden Resolution Testing

This section runs a small synthetic clustering grid. Cluster labels are stored in `adata_cluster.obs`, Scanpy neighbor graphs are stored under explicit keys such as `6neig`, and each neighbor setting gets its own UMAP coordinates in `adata_cluster.obsm`.

In [ ]:
def _plot_categorical_embedding(ax, coords, labels, title, point_size=8):
    cats = pd.Categorical(pd.Series(labels, dtype="string"))
    cmap = plt.get_cmap("tab20")
    for idx, cat in enumerate(cats.categories):
        mask = np.asarray(cats == cat)
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=point_size,
            color=cmap(idx % 20),
            linewidths=0,
            alpha=0.85,
            label=str(cat),
        )
    ax.set_title(title)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_xticks([])
    ax.set_yticks([])
    if 1 < len(cats.categories) <= 6:
        ax.legend(frameon=False, fontsize=5, loc="best", markerscale=1.4)


try:
    import scanpy as sc

    if "adata_norm" not in globals():
        adata_norm = preprocessing.normalize_log1p(adata_qc, inplace=False)
        adata_norm = preprocessing.select_hvgs(adata_norm, n_top_genes=5, flavor="seurat", subset=False)

    neighbor_values = [6, 12, 24]
    resolution_values = [0.2, 0.5, 1.0]
    adata_cluster, clustering_summary = preprocessing.evaluate_clustering_grid(
        adata_norm,
        resolutions=resolution_values,
        n_neighbors=neighbor_values,
        n_pcs=4,
        cluster_key_prefix="leiden_grid",
        random_state=7,
    )
    clustering_summary = clustering_summary.assign(
        neighbors_key=lambda df: df["neighbors"].map(lambda value: f"{value}neig"),
        umap_key=lambda df: df["neighbors"].map(lambda value: f"X_umap_{value}neig"),
    )
    display(clustering_summary)

    fig_grid, axes_grid = plt.subplots(
        len(neighbor_values),
        len(resolution_values),
        figsize=(3.0 * len(resolution_values), 2.6 * len(neighbor_values)),
        dpi=300,
        squeeze=False,
    )
    for row_idx, n_neighbors in enumerate(neighbor_values):
        neighbors_key = f"{n_neighbors}neig"
        umap_key = f"X_umap_{n_neighbors}neig"
        sc.tl.umap(adata_cluster, neighbors_key=neighbors_key, random_state=7)
        adata_cluster.obsm[umap_key] = adata_cluster.obsm["X_umap"].copy()
        coords_umap = np.asarray(adata_cluster.obsm[umap_key])
        for col_idx, resolution in enumerate(resolution_values):
            cluster_key = f"leiden_grid_{n_neighbors}neig_res{str(resolution).replace('.', '_')}"
            _plot_categorical_embedding(
                axes_grid[row_idx, col_idx],
                coords_umap,
                adata_cluster.obs[cluster_key],
                title=f"{n_neighbors} neighbors, res {resolution}",
            )
    plotting.finalize_figure(fig_grid, save_path=None, show=True)
except Exception as exc:
    print(f"Skipped clustering grid / UMAP examples in this environment: {exc}")

## Distance Analysis And Expression By Distance

The spatial proximity workflow reflects the highlighted spatial-proximity notebook: nearest-neighbor spacing, distance to a target cell class, model-ready distance tables, and binned gene expression curves.

In [ ]:
adata_dist = spatial.compute_nearest_neighbor_distances(adata_qc, inplace=False)
target_mask = adata_dist.obs["cell_type"].eq("Macrophage")

distance_df = spatial.celltype_distance_table(
    adata_dist,
    target_mask=target_mask,
    cell_type_key="cell_type",
    sample_key="sample",
)
distance_df = distance_df.merge(
    adata_dist.obs[["condition", "timepoint", "batch_pair"]],
    left_on="obs_id",
    right_index=True,
    how="left",
)
adata_dist.obs["distance_to_macrophage"] = distance_df.set_index("obs_id")["distance_to_target"]
adata_dist.obs["target_region"] = np.where(target_mask.loc[adata_dist.obs_names], "Macrophage", "Other")

distance_results, distance_diagnostics = spatial.run_celltype_distance_analysis(
    distance_df,
    distance_threshold=250,
)
display(distance_df.head())
display(distance_results.head())

In [ ]:
distance_curve = spatial.compute_var_by_distance(
    adata_dist,
    genes=["Ifng", "Col1a1"],
    anchor_key="target_region",
    group_key="condition",
    distance_key="distance_to_macrophage",
    bins=np.linspace(0, 250, 8),
    layer="counts",
)
display(distance_curve.head())

fig_dist, ax_dist = plt.subplots(figsize=(4, 3), dpi=300)
for (gene, condition), sub in distance_curve.groupby(["gene", "condition"], observed=True):
    mid = sub["distance_bin"].apply(lambda value: value.mid)
    ax_dist.plot(mid, sub["expression"], marker="o", label=f"{gene}, {condition}")
ax_dist.set_xlabel("Distance to macrophage")
ax_dist.set_ylabel("Mean expression")
ax_dist.legend(frameon=False, fontsize=6)
plotting.finalize_figure(fig_dist, save_path=None, show=True)

## Squidpy Spatial Graphs, Moran's I, Co-Occurrence, And Spatial Leiden

These wrappers call Squidpy and spatialleiden only when those optional packages are installed. If they are absent, the cell explains the skip instead of failing silently.

In [ ]:
try:
    adata_sq = spatial.compute_spatial_neighbors(
        adata_qc,
        coord_type="generic",
        n_neighs=6,
        key_added="spatial",
        inplace=False,
    )
    moran = spatial.compute_spatial_autocorrelation(
        adata_sq,
        genes=["Ifng", "Col1a1"],
        mode="moran",
        connectivity_key="spatial_connectivities",
    )
    display(moran.head())

    co_occurrence = spatial.compute_co_occurrence(
        adata_sq,
        cluster_key="cell_type",
        source="Macrophage",
        interval=np.arange(0, 120, 20),
    )
    display(co_occurrence.head())
except ImportError as exc:
    adata_sq = adata_qc.copy()
    print(f"Skipped optional Squidpy spatial graph/autocorrelation examples: {exc}")

In [ ]:
try:
    adata_local_moran = spatial.compute_local_moran(
        adata_sq,
        genes=["Ifng", "Col1a1"],
        graph_name="spatial",
        inplace=False,
    )
    display(adata_local_moran.obsm["local_moran"].head())
except (ImportError, NotImplementedError, KeyError, RuntimeError) as exc:
    print(f"Skipped optional local Moran's I example: {exc}")

try:
    adata_spatial_leiden = preprocessing.run_spatial_leiden(
        adata_sq,
        spatial_connectivities_key="spatial_connectivities",
        key_added="spatial_leiden_mock",
        resolution=0.4,
    )
    display(adata_spatial_leiden.obs["spatial_leiden_mock"].value_counts())
except Exception as exc:
    print(f"Skipped optional spatial Leiden example: {exc}")

## Lesion/Niche Analysis

This section mirrors the lesion niche workflow: spatial-neighbor coherence, CellCharter-style embeddings, niche labels, composition tables, and cosine similarity to reference states.

In [ ]:
adata_niche = adata_qc.copy()
adata_niche.obs["niche_mock"] = pd.Categorical(np.where(adata_niche.obs["lesion"], "lesion_edge", "background"))
adata_niche.obsp["mock_connectivities"] = sparse.eye(adata_niche.n_obs, format="csr")

niche_metrics = niches.evaluate_niche_result(
    adata_niche,
    label_key="niche_mock",
    embedding_key="X_mock_niche",
    conn_key="mock_connectivities",
    cell_type_key="cell_type",
)
niche_composition = niches.niche_composition_table(
    adata_niche,
    niche_key="niche_mock",
    group_key="cell_type",
)
similarity = niches.cosine_similarity_centroids(
    adata_niche,
    query_groups=["Tumor", "Macrophage"],
    reference_groups=["T cell", "Hepatocyte"],
    group_col="cell_type",
    layer="counts",
)

display(niche_metrics)
display(niche_composition)
display(similarity)

fig_niche_heatmap, ax_niche_heatmap = plotting.plot_heatmap(similarity, save_path=None, show=True)

## Cell Composition And Report Tables

Report and table helpers return DataFrames and only save when a path is explicitly provided. Here, all save paths remain `None`.

In [ ]:
celltype_composition = composition.celltype_stacked_table(
    adata_qc,
    sample_col="sample",
    celltype_col="cell_type",
)
sample_report = reports.sample_summary(adata_qc, sample_col="sample", save_path=None)
report_bundle = reports.table_with_metadata(sample_report, qc_metadata, save_path=None)

display(celltype_composition.head())
display(report_bundle["table"])
display(report_bundle["metadata"])

## scCODA Composition Workflow

This optional example prepares a sample-by-cell-type count matrix for scCODA and initializes a small compositional model when `sccoda` is installed. Posterior sampling is disabled by default because scCODA depends on the TensorFlow modeling stack and can be slow in lightweight environments.

In [ ]:
sccoda_obs = adata_qc.obs[["sample", "condition", "cell_type"]].copy()
sccoda_obs["sccoda_sample"] = (
    sccoda_obs["sample"].astype(str) + "_" + sccoda_obs["condition"].astype(str)
)
sccoda_covariates = (
    sccoda_obs[["sccoda_sample", "sample", "condition"]]
    .drop_duplicates(subset="sccoda_sample")
    .set_index("sccoda_sample")
)
sccoda_counts = pd.crosstab(sccoda_obs["sccoda_sample"], sccoda_obs["cell_type"])
sccoda_input = sccoda_covariates.join(sccoda_counts, how="left").fillna(0)
count_columns = sccoda_counts.columns.tolist()
sccoda_input[count_columns] = sccoda_input[count_columns].astype(int)
display(sccoda_input)

sccoda_proportions = sccoda_counts.div(sccoda_counts.sum(axis=1), axis=0)
fig_sccoda_input, ax_sccoda_input = plt.subplots(figsize=(5.0, 3.0), dpi=300)
sccoda_proportions.plot(kind="bar", stacked=True, ax=ax_sccoda_input, width=0.8)
ax_sccoda_input.set_xlabel("Synthetic sample")
ax_sccoda_input.set_ylabel("Cell-type proportion")
ax_sccoda_input.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.02, 1), loc="upper left")
plotting.finalize_figure(fig_sccoda_input, save_path=None, show=True)

try:
    import inspect
    from sccoda.util import cell_composition_data as ccd
    from sccoda.util import comp_ana as ca

    sccoda_data = ccd.from_pandas(
        sccoda_input,
        covariate_columns=["condition", "sample"],
    )
    sccoda_reference = "Hepatocyte" if "Hepatocyte" in count_columns else count_columns[0]
    sccoda_model = ca.CompositionalAnalysis(
        sccoda_data,
        formula="condition",
        reference_cell_type=sccoda_reference,
    )
    print(f"Initialized scCODA model with reference cell type: {sccoda_reference}")

    RUN_SCCODA_HMC = False
    if RUN_SCCODA_HMC:
        sample_hmc_kwargs = {}
        sample_hmc_signature = inspect.signature(sccoda_model.sample_hmc)
        if "num_results" in sample_hmc_signature.parameters:
            sample_hmc_kwargs["num_results"] = 200
        if "num_burnin" in sample_hmc_signature.parameters:
            sample_hmc_kwargs["num_burnin"] = 100
        if "num_warmup" in sample_hmc_signature.parameters:
            sample_hmc_kwargs["num_warmup"] = 100
        if "chains" in sample_hmc_signature.parameters:
            sample_hmc_kwargs["chains"] = 1
        elif "n_chains" in sample_hmc_signature.parameters:
            sample_hmc_kwargs["n_chains"] = 1
        if "target_accept_prob" in sample_hmc_signature.parameters:
            sample_hmc_kwargs["target_accept_prob"] = 0.8
        if "target_accept" in sample_hmc_signature.parameters:
            sample_hmc_kwargs["target_accept"] = 0.8

        sccoda_result = sccoda_model.sample_hmc(**sample_hmc_kwargs)
        if hasattr(sccoda_result, "set_fdr"):
            sccoda_result.set_fdr(est_fdr=0.2)
        if hasattr(sccoda_result, "effect_df"):
            display(sccoda_result.effect_df.head())
    else:
        print("Set RUN_SCCODA_HMC=True in this cell to run a small synthetic scCODA posterior sample.")
except ImportError as exc:
    print(f"Skipped optional scCODA example: {exc}")
except Exception as exc:
    print(f"Skipped optional scCODA example after importing sccoda: {exc}")

## Differential Expression, GSEA-Style Tables, Dotplots, Heatmaps, And Volcano Plots

The examples below use mock DE and enrichment results. They demonstrate table formatting and plotting without running PyDESeq2, gseapy, or decoupler on real data.

In [ ]:
pseudobulk = de.make_pseudobulk_replicates(
    adata_qc,
    group_cols=["sample", "condition"],
    layer="counts",
    min_cells=2,
)
display(pseudobulk.head())

mock_de = pd.DataFrame(
    {
        "gene": genes,
        "log2FoldChange": rng.normal(0, 1, len(genes)),
        "padj": np.linspace(0.001, 0.2, len(genes)),
        "score": rng.normal(0, 2, len(genes)),
    }
)
fig_volcano, ax_volcano = plotting.plot_volcano(mock_de, save_path=None, show=True)

mock_gsea = pd.DataFrame(
    {
        "Term": ["HALLMARK_INTERFERON_GAMMA_RESPONSE", "REACTOME_EXTRACELLULAR_MATRIX_ORGANIZATION"],
        "Lead_genes": ["Ifng;GeneA;GeneB", "Col1a1;GeneC"],
        "NES": [1.8, -1.3],
        "FDR q-val": [0.01, 0.04],
    }
)
leading_edge = enrichment.leading_edge_table(mock_gsea)
display(leading_edge)

dot_df = mock_gsea.assign(
    pathway=mock_gsea["Term"].map(enrichment.clean_term_label),
    neg_log10_fdr=-np.log10(mock_gsea["FDR q-val"]),
)
fig_dot, ax_dot = plotting.plot_dotplot(
    dot_df,
    x="NES",
    y="pathway",
    size="neg_log10_fdr",
    color="NES",
    save_path=None,
    show=True,
)

heatmap_df = pd.DataFrame(rng.normal(size=(4, 5)), index=["Tumor", "T cell", "Macrophage", "Hepatocyte"], columns=genes[:5])
fig_heatmap, ax_heatmap = plotting.plot_heatmap(heatmap_df, save_path=None, show=True)

## Ligand-Receptor Summary Workflow

This synthetic example mirrors the LR table standardization and top-pair ranking workflow from the highlighted ligand-receptor notebooks. It does not run stLearn or write LR result files.

In [ ]:
lr_raw = pd.DataFrame(
    {
        "ligand": ["Ccl8", "Cd84", "Ifng", "Col1a1"],
        "receptor": ["Ccr5", "Cd84", "Ifngr1", "Itga1"],
        "lr_score": [0.8, 0.5, 0.6, 0.3],
        "n_spots": [40, 18, 25, 10],
        "condition": ["ctrl", "ctrl", "tx", "tx"],
    }
)
lr_standardized = lr.standardize_lr_table(lr_raw)
lr_ranked = lr.rank_lr_pairs(lr_standardized, condition_col="condition", top_n=2, save_path=None)
display(lr_standardized)
display(lr_ranked)

lr_spatial_df = spatial_df.copy()
lr_spatial_df["lr_scores"] = rng.random(len(lr_spatial_df))
fig_lr, ax_lr = lr.plot_spatial_lr_analysis(lr_spatial_df, score_col="lr_scores", save_path=None, show=True)

## CODEX And H&E Alignment With Mock Landmarks

The reference workflows align CODEX, H&E, masks, labels, and SpatialData objects. This demonstration uses tiny in-memory landmarks and masks only. It does not read or save real images or SpatialData stores.

In [ ]:
xenium_landmarks = np.array([[5, 5], [30, 8], [8, 32], [32, 30]], dtype=float)
he_landmarks = np.array([[8, 10], [34, 16], [4, 36], [29, 40]], dtype=float)
codex_landmarks = np.array([[7, 6], [31, 11], [10, 35], [34, 33]], dtype=float)

xenium_to_he = alignment.align_xenium_to_he(
    xenium_landmarks=xenium_landmarks,
    he_landmarks=he_landmarks,
    xenium_coordinates=spatial_df[["x", "y"]].to_numpy(),
    transform_type="affine",
)
xenium_to_codex = alignment.align_xenium_to_codex(
    xenium_landmarks=xenium_landmarks,
    codex_landmarks=codex_landmarks,
    transform_type="rigid",
)
display(xenium_to_he["transform"])
display({"he_rms_error": xenium_to_he["rms_error"], "codex_rms_error": xenium_to_codex["rms_error"]})

fig_align, ax_align = alignment.plot_alignment_qc(
    reference_coords=he_landmarks,
    moving_coords=xenium_landmarks,
    transform=xenium_to_he["transform"],
    title="Synthetic Xenium to H&E landmarks",
    save_path=None,
    show=True,
)

mock_mask = np.zeros((40, 40), dtype=np.uint8)
mock_mask[10:25, 12:30] = 1
mock_shifted = np.roll(mock_mask, shift=(3, -4), axis=(0, 1))
mock_overlay = alignment.overlay_images(mock_mask, mock_shifted)

fig_mask, ax_mask = plt.subplots(figsize=(3, 3), dpi=300)
ax_mask.imshow(mock_overlay, interpolation="nearest")
ax_mask.set_title("Mock CODEX/H&E overlay")
ax_mask.axis("off")
plotting.finalize_figure(fig_mask, save_path=None, show=True)

try:
    mock_sdata = image.binary_mask_to_spatialdata(mock_mask, name="mock_mask")
    print(type(mock_sdata))
except ImportError as exc:
    print(f"Skipped optional SpatialData mask object example: {exc}")

## Final Checks

Every plot above uses the default display behavior and returns figure/axis objects. No cell provides a save path, writes an output file, or loads real project data. Optional package examples are guarded so the notebook can still serve as documentation in a lightweight environment.